In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('dados_passagens_raw.csv')
df.head()


,compra_id,quantidade_assentos,posicoes_assentos,origem,destino,preco_unitario,data_partida
0,C-000395,3,"9A, 11A, 15B",São Paulo (Tietê),Salvador (BA),270.97,2024/07/13
1,C-000270,2,"18A, 17B",São Paulo (Tietê),Curitiba (PR),114.81,2024-06-23 07:30
2,C-000094,3,"19B, 17B, 3A",São Paulo (Tietê),Porto Alegre (RS),221.01,2024-10-01 23:45
3,C-000123,4,"16B, 17A, 9B, 3B",São Paulo (Tietê),Rio de Janeiro (RJ),112.85,2024-04-24 18:00
4,C-000308,2,"5B, 10A",São Paulo (Tietê),Fortaleza (CE),317.82,2024-05-17 20:00


In [3]:
# Vamos começar o tratamento de correção de erros
# Vamos verificar os valores nulos por coluna

df.isnull().sum()


compra_id               0
quantidade_assentos     0
posicoes_assentos      31
origem                  0
destino                21
preco_unitario         29
data_partida           36
dtype: int64

In [4]:
df.describe(include='all')

,compra_id,quantidade_assentos,posicoes_assentos,origem,destino,preco_unitario,data_partida
count,511,511.000000,480,511,490,482.000000,475
unique,500,NaN,292,2,16,NaN,75
top,C-000365,NaN,15B,São Paulo (Tietê),Brasília (DF),NaN,2024-05-22 15:30
freq,2,NaN,10,508,68,NaN,18
mean,NaN,1.921722,NaN,NaN,NaN,248.444274,NaN
std,NaN,1.078187,NaN,NaN,NaN,660.905360,NaN
min,NaN,0.000000,NaN,NaN,NaN,-41.690000,NaN
25%,NaN,1.000000,NaN,NaN,NaN,121.570000,NaN
50%,NaN,2.000000,NaN,NaN,NaN,185.150000,NaN
75%,NaN,3.000000,NaN,NaN,NaN,270.985000,NaN


In [5]:
# 1. Erros de digitação no destino

valores_unicos = df['destino'].unique()
valores_unicos

<StringArray>
[      'Salvador (BA)',       'Curitiba (PR)',   'Porto Alegre (RS)',
 'Rio de Janeiro (RJ)',      'Fortaleza (CE)',       'Brasília (DF)',
  'Florianópolis (SC)',         'Recife (PE)',       'Brasíli a(DF)',
        'Goiânia (GO)',       'brasília (df)', 'Belo Horizonte (MG)',
                   nan,    'Porto Alegre(RS)',   'Belo Horizonte MG',
       'Brasilia (DF)',       'Brasílai (DF)']
Length: 17, dtype: str

In [6]:
erros = ['Brasíli a(DF)', 'brasília (df)', 'Brasilia (DF)', 'Brasílai (DF)']
df['destino'] = df['destino'].replace(erros, 'Brasília (DF)')

df['destino'] = df['destino'].replace('Belo Horizonte MG', 'Belo Horizonte (MG)')

df['destino'] = df['destino'].replace('Porto Alegre(RS)', 'Porto Alegre (RS)')

valores_unicos = df['destino'].unique()
valores_unicos


<StringArray>
[      'Salvador (BA)',       'Curitiba (PR)',   'Porto Alegre (RS)',
 'Rio de Janeiro (RJ)',      'Fortaleza (CE)',       'Brasília (DF)',
  'Florianópolis (SC)',         'Recife (PE)',        'Goiânia (GO)',
 'Belo Horizonte (MG)',                   nan]
Length: 11, dtype: str

In [7]:
# 2. Erros de destino nulo

linhas_vazias = df[df['destino'].isna() | (df['destino'].astype(str).str.strip() == '')]
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(linhas_vazias)



,compra_id,quantidade_assentos,posicoes_assentos,origem,destino,preco_unitario,data_partida
59,C-000172,2,"9A, 13B",São Paulo (Tietê),NaN,NaN,2024-05-05 11:00
71,C-000106,2,"12A, 10B",São Paulo (Tietê),NaN,185.15,2024-03-12 15:30
85,C-000481,2,"4B, 1A",São Paulo (Tietê),NaN,133.63,2024-01-04 15:30
86,C-000246,1,8A,São Paulo (Tietê),NaN,121.51,2024-04-29 22:30
100,C-000040,4,"7B, 11A, 8B, 9A",São Paulo (Tietê),NaN,-33.85,2024-04-27 11:00
113,C-000090,2,"3A, 16A",São Paulo (Tietê),NaN,152.81,2024-11-01 22:30
175,C-000355,3,"18B, 8B, 4A",São Paulo (Tietê),NaN,133.63,2024-01-04 15:30
178,C-000367,2,NaN,São Paulo (Tietê),NaN,NaN,NaN
200,C-000419,2,"15B, 14B",São Paulo (Tietê),NaN,114.81,2024-06-23 07:30
272,C-000284,2,"4B, 3B",São Paulo (Tietê),NaN,322.95,2024-03-24 15:30


In [8]:
# Preenche os NaNs da coluna 'destino' usando um destino válido que coincide com 'data_partida'

df['destino'] = df['destino'].fillna(df.groupby('data_partida')['destino'].transform('first'))
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(df)


,compra_id,quantidade_assentos,posicoes_assentos,origem,destino,preco_unitario,data_partida
0,C-000395,3,"9A, 11A, 15B",São Paulo (Tietê),Salvador (BA),270.97,2024/07/13
1,C-000270,2,"18A, 17B",São Paulo (Tietê),Curitiba (PR),114.81,2024-06-23 07:30
2,C-000094,3,"19B, 17B, 3A",São Paulo (Tietê),Porto Alegre (RS),221.01,2024-10-01 23:45
3,C-000123,4,"16B, 17A, 9B, 3B",São Paulo (Tietê),Rio de Janeiro (RJ),112.85,2024-04-24 18:00
4,C-000308,2,"5B, 10A",São Paulo (Tietê),Fortaleza (CE),317.82,2024-05-17 20:00
5,C-000365,2,"10B, 21B",São Paulo (Tietê),Rio de Janeiro (RJ),97.45,2024-01-13 22:30
6,C-000208,4,"20A, 21B, 2A, 3A",São Paulo (Tietê),Brasília (DF),185.15,2024-03-12 15:30
7,C-000359,2,"21B, 18A",São Paulo (Tietê),Curitiba (PR),108.82,2024-04-20 23:45
8,C-000068,1,3A,São Paulo (Tietê),Brasília (DF),174.22,2024-10-02 22:30
9,C-000451,3,"12A, 10A, 3B",São Paulo (Tietê),Fortaleza (CE),338.27,2024-06-10 15:30


In [9]:
# Verificando quantos destinos nulos ainda existem para preenhcer os restantes

linhas_vazias = df[df['destino'].isna() | (df['destino'].astype(str).str.strip() == '')]
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(linhas_vazias)

,compra_id,quantidade_assentos,posicoes_assentos,origem,destino,preco_unitario,data_partida
178,C-000367,2,NaN,São Paulo (Tietê),NaN,NaN,NaN
470,C-000198,1,18B,São Paulo (Tietê),NaN,317.82,17/05/2024


In [10]:
# Como a linha 178 está praticamente toda nula, vamos dropá-la. Já a linha 470, pelo valor e data, podemos preencher com o destino correto, que é "Fortaleza (CE)" e aproveitar para corrigir a data para "2024-05-17 20:00".

df.loc[470, ['destino', 'data_partida']] = ['Fortaleza (CE)', '2024-05-17 20:00']
display(df.loc[[470]])


,compra_id,quantidade_assentos,posicoes_assentos,origem,destino,preco_unitario,data_partida
470,C-000198,1,18B,São Paulo (Tietê),Fortaleza (CE),317.82,2024-05-17 20:00


In [11]:
# 3. Erros de preco_unitario

linhas_vazias = df[
    df['preco_unitario'].isna() | 
    (df['preco_unitario'] <= 0) | 
    (df['preco_unitario'] > 400)
]
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(linhas_vazias)


,compra_id,quantidade_assentos,posicoes_assentos,origem,destino,preco_unitario,data_partida
11,C-000086,2,"6A, 3B",São Paulo (Tietê),Florianópolis (SC),NaN,2024-12-15 09:00
36,C-000432,1,16A,São Paulo (Tietê),Brasília (DF),0.00,2024-10-02 22:30
50,C-000235,3,"9A, 3B, 16A",São Paulo (Tietê),Brasília (DF),NaN,2024-03-12 15:30
55,C-000379,1,2B,São Paulo (Tietê),Porto Alegre (RS),NaN,2024-07-12 07:30
59,C-000172,2,"9A, 13B",São Paulo (Tietê),Recife (PE),NaN,2024-05-05 11:00
77,C-000310,2,"4A, 7B",São Paulo (Tietê),Porto Alegre (RS),-36.85,2024-10-01 23:45
100,C-000040,4,"7B, 11A, 8B, 9A",São Paulo (Tietê),Fortaleza (CE),-33.85,2024-04-27 11:00
122,C-000211,1,13B,São Paulo (Tietê),Curitiba (PR),NaN,2024-01-04 15:30
137,C-000199,2,"11B, 7A",São Paulo (Tietê),Brasília (DF),NaN,2024-03-12 15:30
140,C-000249,3,"21B, 8B, 9A",São Paulo (Tietê),Belo Horizonte (MG),NaN,2024-09-15 18:00


In [12]:
# Converter os valores discrepantes para NaN e trocar para o valor válido com mesmo 'destino' e 'data_partida'

df.loc[(df['preco_unitario'] <= 0) | (df['preco_unitario'] > 400), 'preco_unitario'] = np.nan

df['preco_unitario'] = df['preco_unitario'].fillna(
    df.groupby(['destino', 'data_partida'])['preco_unitario'].transform('first')
)

linhas_vazias = df[df['preco_unitario'].isna()]
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(linhas_vazias)
    

,compra_id,quantidade_assentos,posicoes_assentos,origem,destino,preco_unitario,data_partida
178,C-000367,2,NaN,São Paulo (Tietê),NaN,NaN,NaN


In [13]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df)


,compra_id,quantidade_assentos,posicoes_assentos,origem,destino,preco_unitario,data_partida
0,C-000395,3,"9A, 11A, 15B",São Paulo (Tietê),Salvador (BA),270.97,2024/07/13
1,C-000270,2,"18A, 17B",São Paulo (Tietê),Curitiba (PR),114.81,2024-06-23 07:30
2,C-000094,3,"19B, 17B, 3A",São Paulo (Tietê),Porto Alegre (RS),221.01,2024-10-01 23:45
3,C-000123,4,"16B, 17A, 9B, 3B",São Paulo (Tietê),Rio de Janeiro (RJ),112.85,2024-04-24 18:00
4,C-000308,2,"5B, 10A",São Paulo (Tietê),Fortaleza (CE),317.82,2024-05-17 20:00
5,C-000365,2,"10B, 21B",São Paulo (Tietê),Rio de Janeiro (RJ),97.45,2024-01-13 22:30
6,C-000208,4,"20A, 21B, 2A, 3A",São Paulo (Tietê),Brasília (DF),185.15,2024-03-12 15:30
7,C-000359,2,"21B, 18A",São Paulo (Tietê),Curitiba (PR),108.82,2024-04-20 23:45
8,C-000068,1,3A,São Paulo (Tietê),Brasília (DF),174.22,2024-10-02 22:30
9,C-000451,3,"12A, 10A, 3B",São Paulo (Tietê),Fortaleza (CE),338.27,2024-06-10 15:30


In [14]:
# 4. Tratamento da coluna data_partida
#  Converte a coluna 'data_partida' para datetime YYYY-MM-DD HH:mm e deixa os registros inválidos como NaT (Not a Time) 
data_convertida = pd.to_datetime(df['data_partida'], format='%Y-%m-%d %H:%M', errors='coerce')

df['data_partida'] = data_convertida.dt.strftime('%Y-%m-%d %H:%M')

# Preenche os NaNs buscando a primeira data válida para a mesma combinação de 'destino' e 'preco_unitario'
df['data_partida'] = df['data_partida'].fillna(
    df.groupby(['destino', 'preco_unitario'])['data_partida'].transform('first')
)

linhas_vazias = df[df['data_partida'].isna()]
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(linhas_vazias)


,compra_id,quantidade_assentos,posicoes_assentos,origem,destino,preco_unitario,data_partida
54,C-000469,2,"16B, 11A",São Paulo (Tietê),Porto Alegre (RS),202.74,NaN
178,C-000367,2,NaN,São Paulo (Tietê),NaN,NaN,NaN
199,C-000464,1,20B,São Paulo (Tietê),Porto Alegre (RS),202.74,NaN


In [1]:
linhas_vazias = df[df['quantidade_assentos'] == 0]
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(linhas_vazias)

NameError: name 'df' is not defined

In [48]:
df.isnull().sum()


compra_id               0
quantidade_assentos     0
posicoes_assentos      31
origem                  0
destino                 1
preco_unitario          1
data_partida            3
dtype: int64

In [ ]:
# Após o tratamento até aqui, vamos manter os dados das vendas em que não houve escolha de assento (pode ocorrer dessa escolha não ter sido feita ou ignorada). Mas vamos dropar as demais linhas que não pudemos identificar.
# Ademais, vamos dropar as linhas que apresentem duplicatas exatas em todos os campos, pois segundo nossas regras de preenchimento, há forte indício que são registros duplicados de uma mesma venda.
# Após os drops, vou converter a coluna data para datetime.

df.drop(index=[54, 178, 199], inplace=True)
df.drop_duplicates(inplace=True)
df['data_partida'] = pd.to_datetime(df['data_partida'])
df['preco_unitario'] = pd.to_numeric(df['preco_unitario'], errors='coerce').round(2)

df.isnull().sum()

df.info()


<class 'pandas.DataFrame'>
Index: 497 entries, 0 to 510
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   compra_id            497 non-null    str           
 1   quantidade_assentos  497 non-null    int64         
 2   posicoes_assentos    467 non-null    str           
 3   origem               497 non-null    str           
 4   destino              497 non-null    str           
 5   preco_unitario       497 non-null    float64       
 6   data_partida         497 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 47.2 KB


In [55]:
# 5. Criação da coluna 'receita'
df['receita'] = (df['quantidade_assentos'] * df['preco_unitario'].astype(float)).round(2)

# Receita Mês a Mês 
receita_mensal = (
    df.groupby(df['data_partida'].dt.to_period('M'))['receita']
    .sum()
    .round(2)
    .reset_index()
)
receita_mensal.columns = ['Mês/Ano', 'Receita Total (R$)']
receita_mensal['Mês/Ano'] = receita_mensal['Mês/Ano'].astype(str)

print("=== RECEITA MÊS A MÊS ===")
display(receita_mensal)

# Receita por Destino (ordenada da maior para a menor receita)
receita_destino = (
    df.groupby('destino')['receita']
    .sum()
    .round(2)
    .sort_values(ascending=False)
    .reset_index()
)
receita_destino.columns = ['Destino', 'Receita Total (R$)']

print("\n=== RECEITA POR DESTINO ===")
display(receita_destino)



=== RECEITA MÊS A MÊS ===


,Mês/Ano,Receita Total (R$)
0,2024-01,15131.90
1,2024-02,11805.55
2,2024-03,25840.76
3,2024-04,19857.43
4,2024-05,39538.79
5,2024-06,8633.16
6,2024-07,9984.98
7,2024-08,8547.42
8,2024-09,4413.84
9,2024-10,14780.15



=== RECEITA POR DESTINO ===


,Destino,Receita Total (R$)
0,Fortaleza (CE),29195.31
1,Recife (PE),25985.12
2,Brasília (DF),24977.83
3,Salvador (BA),23982.92
4,Florianópolis (SC),20135.07
5,Porto Alegre (RS),16265.51
6,Curitiba (PR),15871.84
7,Goiânia (GO),13178.73
8,Belo Horizonte (MG),10327.99
9,Rio de Janeiro (RJ),7167.40


In [56]:
# Salva o DataFrame tratado em um arquivo CSV
df.to_csv('dados_passagens_limpo.csv', index=False, encoding='utf-8-sig', sep=';')
